In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
spark = SparkSession.builder.getOrCreate()
order_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true") 
    .load("/Volumes/orders/default/orders_csv/orders_100mb.csv")
)
order_df.show(5)
order_df.printSchema()

In [0]:
product_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true") 
    .load("/Volumes/orders/default/orders_csv/products.csv")
)
product_df.show(5)


Q1) Using the orders and products datasets, find the **top 2 most profitable products in each state**, where profit is defined as `(unit_price − cost_price) * quantity`. Consider only delivered orders. Return state, product_id, total profit, and rank within the state.


In [0]:
order_df_filtered = (
    order_df
    .filter(F.col("order_status") == 'DELIVERED')
    .select("product_id", "state", "price", "quantity")
)
product_df_filtered = product_df.select(F.col("product_id"), F.col("cost_price"))

df_merged = (
    order_df_filtered.join(F.broadcast(product_df_filtered), "product_id")
    .withColumn("profit",(F.col("price") - F.col("cost_price"))*F.col("quantity"))
    .groupby(F.col("state"),F.col("product_id"))
    .agg(
        F.sum(F.col("profit")).alias("total_profit")
    )
)
w = Window.partitionBy(F.col("state")).orderBy(F.col("total_profit").desc())

df_merged = (
    df_merged.withColumn("rank_within_state",F.row_number().over(w))
    .filter(F.col("rank_within_state") < 3)
)
df_merged.show()

Q2) For each customer, identify their **most recent order** and calculate the **difference between that order’s unit price and the average unit price of all products purchased by that customer**. Return customer_id, order_id, order_date, unit_price, and the price difference.

In [0]:
w = Window.partitionBy("customer_id").orderBy(F.col("order_date").desc(), F.col("order_id").desc())

order_df_filtered = (
          order_df.select("customer_id","order_id","order_date",'price')
          .withColumn("customer_ranked", F.row_number().over(w))
          .filter(F.col("customer_ranked") == 1)
)
avg_price_df = (
          order_df.groupby('customer_id')
          .agg(
            F.avg("price").alias("avg_unit_price")
          )
)
df_merged  = (
        order_df_filtered.join(avg_price_df,"customer_id","inner")
        .withColumn("price_difference",F.col("price") - F.col("avg_unit_price"))
        .select("customer_id","order_id","order_date","price","price_difference")
)
df_merged.show()

#m2

In [0]:
w_latest = Window.partitionBy("customer_id") \
                 .orderBy(F.col("order_date").desc(), F.col("order_id").desc())

w_avg = Window.partitionBy("customer_id")

df = (
    order_df
    .withColumn("avg_unit_price", F.avg("price").over(w_avg))
    .withColumn("rank", F.row_number().over(w_latest))
    .filter(F.col("rank") == 1)
    .withColumn("price_difference", F.col("price") - F.col("avg_unit_price"))
    .select("customer_id", "order_id", "order_date", "price", "price_difference")
)

df.show()

Q3)Using orders and products data, determine for each year the **product that contributed the highest percentage of total yearly revenue**. Return order_year, product_id, product_revenue, total_year_revenue, and revenue_percentage.

In [0]:
w = Window.partitionBy("order_year")
w_rank = Window.partitionBy("order_year").orderBy(F.col("revenue_percentage").desc())
df = (
    order_df
    .withColumn("product_revenue", F.col("quantity")* F.col("price"))
    .withColumn("order_year",F.year(F.col("order_date")))
    .select("order_year",'product_id','product_revenue')
    .groupby(["order_year",'product_id'])
    .agg(
      F.sum(F.col("product_revenue")).alias("product_revenue")
    )
    .withColumn('total_year_revenue',F.sum("product_revenue").over(w))
    .withColumn("revenue_percentage", F.col("product_revenue")*100.0/F.col("total_year_revenue"))
    .withColumn("ranked",F.row_number().over(w_rank))
    .filter(F.col("ranked")== 1)
    .select("order_year",'product_id',"product_revenue","total_year_revenue","revenue_percentage")
)
df.show()

Q4. Using the orders and products datasets, calculate for each state the running cumulative revenue over time, ordered by order_date, and return state, order_date, order_id, order_value, and cumulative revenue.

In [0]:
window_cum_sum = Window.partitionBy("state").orderBy(F.col("order_date"))\
    .rowsBetween(Window.unboundedPreceding,Window.currentRow)
df = (
    order_df.withColumn("order_value", F.col("quantity")* F.col("price"))
    .withColumn("cumulative_revenue",F.sum("order_value").over(window_cum_sum))
    .select("order_date",'state',"order_date","order_id","order_value","cumulative_revenue")
    .orderBy("state", "order_date")
)
df.show()

Q5. For each product, find the first and last order date along with the total revenue generated between those dates, considering only non cancelled orders.

In [0]:
window = Window.partitionBy("product_id")
df = (
    order_df.filter(F.col("order_status") != 'CANCELLED')
    .join(F.broadcast(product_df), 'product_id','right')
    .withColumn("revenue", F.col("quantity")* F.col("price"))
    .withColumn("last_order_date",F.max("order_date").over(window))   
    .withColumn("first_order_date",F.min("order_date").over(window))  
    .withColumn(
        "total_revenue",
        F.when(
            F.col("order_date").between(
                F.col("first_order_date"),
                F.col("last_order_date")
            ),
            F.col("revenue")
        ).otherwise(None)
    )
    .groupBy(["product_id","first_order_date","last_order_date"])
    .agg(
        F.sum("total_revenue").alias("total_revenue")
    )
)
df.show()

Q6. Identify customers whose latest order value is greater than their previous order value, and return customer_id, order_id, order_date, previous_order_value, and latest_order_value.

In [0]:
w_lag = Window.partitionBy("customer_id").orderBy('order_date')
w = Window.partitionBy("customer_id").orderBy(F.col('order_date').desc())
df = (
    order_df.
    withColumn("prev_price", F.lag(F.col('price')).over(w_lag))
    .withColumn("order_ranked",F.row_number().over(w))
    .filter(
        (F.col("order_ranked") == 1) &
        (F.col("price") > F.col("prev_price"))
    )
    .select("customer_id", "order_id", "order_date", "prev_price", "price")
    
)
df.show()

Q7.Using orders and products data, rank products within each year based on total profit, and return only the top 3 products per year.

In [0]:
w = Window.partitionBy("order_year").orderBy(F.col("total_profit").desc())
df = (
    order_df.withColumn("revenue", F.col("quantity")* F.col("price"))
    .withColumn("order_year",F.year(F.col("order_date")))
    .select("product_id",'order_year','revenue','quantity')
    .join(F.broadcast(product_df),'product_id',)
    .withColumn("cogs",F.col("quantity")*F.col("cost_price"))
    .withColumn("total_profit",F.col("revenue")- F.col("cogs"))
    .groupBy("order_year", "product_id", "product_name").agg(
    F.sum("total_profit").alias("total_profit")
    )
    .withColumn("profit_ranked",F.row_number().over(w))
    .filter(F.col("profit_ranked") <= 3)
)
df.show()

25-04-2026

Q8.For each state and order_status combination, calculate the percentage contribution of that combination to the total revenue of that state, and return state, order_status, revenue, and revenue_percentage.

In [0]:
state_revenue_df = (
  order_df.groupby('state')
  .agg(
    F.sum(F.col("quantity") * F.col("price")).alias("state_revenue")
  )
)
df = (
  order_df
  .withColumn("revenue",F.col("quantity") * F.col("price"))
  .groupby(['state','order_status'])
  .agg(
    F.sum(F.col("revenue")).alias("revenue_combo")
  )
  .join(F.broadcast(state_revenue_df),'state','inner')
  .withColumn('revenue_percentage', F.col('revenue_combo')*100.0/F.coalesce("state_revenue",F.lit(0)))
  .select("state","order_status","revenue_combo","revenue_percentage")
  .orderBy("state")
  
)
df.show()

#m2

In [0]:
w = Window.partitionBy("state")
df = (
  order_df
  .withColumn("revenue",F.col("quantity") * F.col("price"))
  .groupby(['state','order_status'])
  .agg(
    F.sum(F.col("revenue")).alias("revenue_combo")
  )
  .withColumn("state_revenue",F.sum("revenue_combo").over(w))
  .withColumn('revenue_percentage', F.col('revenue_combo')*100.0/F.coalesce("state_revenue",F.lit(0)))
  .select("state","order_status","revenue_combo","revenue_percentage")
  
)
df.show()

Q9.For each customer, calculate the moving average of order value over their last 3 orders, ordered by order_date, and return customer_id, order_id, order_date, order_value, and moving_average.

In [0]:
w = Window.partitionBy("customer_id").orderBy("order_date").rowsBetween(-2, 0)

df = (
    order_df
    .withColumn("order_value",F.col("quantity")*F.col("price"))
    .withColumn("moving_average",F.avg("order_value").over(w))
    .select("customer_id","order_id","order_date","order_value","moving_average")
    .orderBy("customer_id","order_date")

)
df.show()

Q10. Using orders and products datasets, find products whose average selling price is consistently higher than cost price across all years, and return product_id, average_unit_price, average_cost_price, and price_difference.

In [0]:
from pyspark.sql.functions import col, year, avg, min as _min

# Step 1: Extract year
df_orders_year = df_orders.withColumn(
    "order_year",
    year(col("order_date"))
)

# Step 2: Join with products
df_joined = df_orders_year.join(df_products, "product_id", "inner")

# Step 3: Avg selling price per product per year
df_yearly = df_joined.groupBy("product_id", "order_year", "cost_price").agg(
    avg("price").alias("avg_selling_price")
)

# Step 4: Check condition (avg_selling_price > cost_price)
df_check = df_yearly.withColumn(
    "is_profitable",
    col("avg_selling_price") > col("cost_price")
)

# Step 5: Ensure condition holds for ALL years
df_valid_products = df_check.groupBy("product_id").agg(
    _min(col("is_profitable").cast("int")).alias("all_years_profitable")
).filter(col("all_years_profitable") == 1)

# Step 6: Get overall averages
df_final = df_joined.groupBy("product_id", "cost_price").agg(
    avg("price").alias("average_unit_price")
)

# Step 7: Join with valid products
df_result = df_final.join(df_valid_products, "product_id", "inner") \
    .withColumn(
        "price_difference",
        col("average_unit_price") - col("cost_price")
    )

# Final output
df_result = df_result.select(
    "product_id",
    "average_unit_price",
    col("cost_price").alias("average_cost_price"),
    "price_difference"
)
